<a href="https://colab.research.google.com/github/achrekarom12/finetuning-gemma/blob/main/gemma_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetuning gemma-4-E2B-it

In [1]:
!pip install -q torch pillow requests torch torchvision

In [2]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, AutoModelForImageTextToText

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "google/gemma-4-E2B-it"

print(f"Device available: {device}")

Device available: cuda


In [14]:
pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-wp9nwrym
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-wp9nwrym
  Resolved https://github.com/huggingface/transformers.git to commit eed95d8c445b8679ba342cffa947a3ed2b8d7fbc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11655128 sha256=4009d4d564e1dc41fb1f7e5fd8676eba08f2b50e39c837f0b92287844f3d4026
  Stored in directory: /tmp/pip-ephem-wheel-cache-891_tt_9/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.7.0
    Uninstalling transformers-5.7.0:
      Successfully uninstalled transformers-5.7.0


In [4]:
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [ ]:
def chat_with_gemma():
    # Initialize chat history
    messages = []
    print("--- Chat started (Type 'exit' to stop) ---")

    while True:
        user_input = input("User: ")
        if user_input.lower() in ["exit", "quit"]:
            break

        # Prepare the message content
        content = [{"type": "text", "text": user_input}]

        # Add to history
        messages.append({"role": "user", "content": content})

        # 2. Process Input
        # Note: apply_chat_template handles the formatting for Gemma-4
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

        inputs = processor(
            text=prompt,
            return_tensors="pt"
        ).to(device)

        # 3. Generate Response
        output = model.generate(**inputs, max_new_tokens=512)

        # Decode only the newly generated tokens
        generated_text = processor.batch_decode(
            output[:, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )[0]

        print(f"\nAssistant: {generated_text}\n")

        # Add assistant response to history
        messages.append({"role": "assistant", "content": [{"type": "text", "text": generated_text}]})

chat_with_gemma()

--- Chat started (Type 'exit' to stop) ---
User: Hey

Assistant: Hi! How can I help you today? 😊

User: Who are you?

Assistant: I am Gemma 4, a Large Language Model developed by Google DeepMind. I am an open weights model.

User: Who trained you?

Assistant: I was trained by Google DeepMind.

User: Tell me a dark joke

Assistant: Why don't skeletons fight each other?

They don't have the guts.

User: No a racist one

Assistant: I'm sorry, but I cannot fulfill that request. I am programmed to be a helpful and harmless AI, and that includes refusing to generate content that is racist or discriminatory.

User: Great, can I fine tune you?

Assistant: As an open weights model, the fact that my weights are publicly available means that researchers and developers *can* fine-tune versions of me for specific tasks or domains.

However, whether *you* can fine-tune me depends entirely on:

1.  **The specific version of Gemma 4 you are using:** Different sizes (like 2B or 7B) and different model 